In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt


## 0. Read Data for Analysis

In [ ]:
df = pd.read_csv('/Users/adigonitodoulou/Desktop/Data_Analysis_Projects/Booking/hotel_bookings.csv')

In [ ]:
type(df)

In [ ]:
df.dtypes

In [ ]:
df.columns

## 1. Data Cleaning


In [ ]:
filter1 = (df['children'] == 0) & (df['adults'] == 0) & (df['babies'] == 0)

In [ ]:
df[filter1]

In [ ]:
df2 = df[~filter1]

In [ ]:
df2.duplicated().sum()

In [ ]:
data = df2.drop_duplicates()

In [ ]:
data.shape

## 2. EDA

## 2.1 Descriptive Analysis

In [ ]:
data[['lead_time', 'total_of_special_requests', 'adr']].describe().T

In [ ]:
data.info(memory_usage='deep')

In [ ]:
for col in ['lead_time', 'total_of_special_requests', 'adr']:
    print('feature name : {}'.format(col) )
    
    for i in range (90,101,1):
        quantile_value = np.quantile(data[col], q = i/100)
        print('{}th quantile value is {}'.format(i,quantile_value))
    print('\n')


## 2.2 Where do the guests come from

In [ ]:
not_cancelled = data[data['is_canceled'] == 0]

In [ ]:
country_wise_data = not_cancelled['country'].value_counts().reset_index()

In [ ]:
country_wise_data.columns = ['country', 'no_of_guests']

In [ ]:
country_wise_data

In [ ]:
import chart_studio.plotly as py
import plotly.graph_objs as go
import plotly.express as px
from plotly.offline import download_plotlyjs, init_notebook_mode, plot, iplot

init_notebook_mode(connected=True)

In [ ]:
map_guest = px.choropleth(data_frame=country_wise_data,
              locations = country_wise_data['country'],
              color = country_wise_data['no_of_guests'],
              hover_name= country_wise_data['country'],
              title = 'Home Country of Guests'
              )

In [ ]:
map_guest.show()

## 2.3 Is there any difference between assigned and reserved room types

In [ ]:
data.columns

In [ ]:
pivot = pd.crosstab(index = data['reserved_room_type'], columns = data['assigned_room_type'], margins = True)

In [ ]:
pivot

In [ ]:
pivot_normalize = pd.crosstab(index = data['reserved_room_type'], columns = data['assigned_room_type'], margins = True, normalize = 'index').round(2).mul(100)

In [ ]:
pivot_normalize

## 2.4 Which market segment has more bookings

In [ ]:
data['market_segment'].value_counts()

In [ ]:
fig = px.pie(
       values = data['market_segment'].value_counts().values,
       names = data['market_segment'].value_counts().index
       )

In [ ]:
fig.show()

## 2.5 Analysing average price per night (ADR) of various room types for all the market segments

In [ ]:
plt.figure(figsize=(20,10))
sns.barplot(x = 'market_segment', y = 'adr', hue='reserved_room_type', data = data)

## 2.6 Total guests arriving on each day

In [ ]:
data.head(1)

In [ ]:
data['arrival_date_month'].unique()

In [ ]:
dict_month = {
    'July' : 7,    'August' : 8, 'September' : 9,   'October' : 10,  'November' : 11,  'December' : 12,
   'January' : 1,  'February' : 2,     'March' : 3,     'April' : 4,       'May' : 5,      'June' : 6
}

In [ ]:
data['arrival_date_month_index'] = data['arrival_date_month'].map(dict_month).copy()

In [ ]:
data.columns

In [ ]:
data[['arrival_date_year', 'arrival_date_month_index', 'arrival_date_day_of_month']]

In [ ]:
data['arrival_date'] = data['arrival_date_year'].astype(str) + '-' + data['arrival_date_month_index'].astype(str) + '-' + data['arrival_date_day_of_month'].astype(str)

In [ ]:
data.head(2)

In [ ]:
data['total_guests'] = (data['adults'] + data['children'] + data['babies'])

In [ ]:
dataNoCancel = data[data['is_canceled'] == 0]

In [ ]:
guest_arrival_series = dataNoCancel.groupby(['arrival_date'])['total_guests'].sum()


In [ ]:
guest_arrival_series.plot(figsize=(10,6))

## 2.7 Analyzing distribution of guest arrival

In [ ]:
guest_arrival_series.values

In [ ]:
sns.displot(guest_arrival_series.values)

In [ ]:
sns.displot(guest_arrival_series.values, kind='kde')


In [ ]:
np.mean(guest_arrival_series.values)

In [ ]:
np.median(guest_arrival_series.values)

In [ ]:
np.std(guest_arrival_series.values)